<a href="https://colab.research.google.com/github/mohamedlibanml/ai-job-market-analysis/blob/main/Milagro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Step 1: Loading and Preparing the Data
First, we load our store performance datasets and clean up the column names by replacing dots (`.`) with underscores (`_`). This prevents any issues when writing mathematical formulas for our model.
```

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load datasets
train = pd.read_csv('/content/train_data.csv')
test = pd.read_csv('/content/test_data.csv')
sites = pd.read_csv('/content/site_const_data-1.csv')

# Standardize column names (replace '.' with '_')
for d in (train, test, sites):
    d.columns = [c.replace('.', '_') for c in d.columns]

print("Data successfully loaded and prepared!")
print(f"Train size: {train.shape}, Test size: {test.shape}, Sites size: {sites.shape}")

Data successfully loaded and prepared!
Train size: (374, 13), Test size: (85, 13), Sites size: (48, 12)


In [ ]:
def fit_eval(vars_):
    """Fit OLS on train; return model, train R2, test R2 (OSR2), total predicted profit of 48 sites ($M)."""
    m = smf.ols(Y + ' ~ ' + ' + '.join(vars_), data=train).fit()
    pred = m.predict(test)
    sse = ((test[Y] - pred)**2).sum()
    sst = ((test[Y] - train[Y].mean())**2).sum()   # baseline = TRAINING mean (out-of-sample R2)
    return m, m.rsquared, 1 - sse/sst, m.predict(sites).sum()/1e6

## Q1 – Why split the data?
Training set = data the model learns from. Test set = 'new' stores held back to check how well the model predicts data it never saw.

In [ ]:
Y = 'annual_profit'
print(f'Training share: {len(train)/(len(train)+len(test)):.1%}')

Training share: 81.5%


## Q2 – Fit Kathleen's original model
We fit an Ordinary Least Squares (OLS) model for `annual_profit` using the predictors `agg_inc`, `sqft`, `col_grad`, and `com60` to obtain the model equation coefficients.

In [ ]:
A_vars = ['agg_inc', 'sqft', 'col_grad', 'com60']
mA = smf.ols(Y + ' ~ ' + ' + '.join(A_vars), data=train).fit()
print(mA.summary())
display(mA.params)

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.786
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     339.1
Date:                Mon, 02 Nov 2026   Prob (F-statistic):          3.73e-122
Time:                        03:06:15   Log-Likelihood:                -5107.5
No. Observations:                 374   AIC:                         1.022e+04
Df Residuals:                     369   BIC:                         1.024e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    8.36e+04   4.11e+04      2.035      0.0

,0
Intercept,83597.092456
agg_inc,0.002807
sqft,383.363054
col_grad,346821.138695
com60,218265.764624


## Q3 – Predict one store
We plug the specific store criteria into our fitted model (Model A) to find its predicted annual profit.

In [ ]:
new_store = pd.DataFrame({
    'agg_inc': [100000000],
    'sqft': [800],
    'col_grad': [0.30],
    'com60': [0.10]
})

pred_profit = mA.predict(new_store)
print(f"Predicted Annual Profit for the store: ${pred_profit.iloc[0]:,.2f}")

Predicted Annual Profit for the store: $796,894.99


## Q4 – Train and test R2
We compute Model A's performance on both the training set (Train $R^2$) and the test set (Test Out-of-Sample $R^2$) to see how well it generalizes to unseen data.

In [ ]:
_, r2_tr, r2_te, totA = fit_eval(A_vars)
print(f'Train R2 = {r2_tr:.4f}   Test R2 = {r2_te:.4f}')

Train R2 = 0.7861   Test R2 = 0.7201


## Q5 – Variable Significance in Model A
A variable is considered statistically significant at the 5% significance level if its $p$-value is less than $0.05$. We extract, drop the `Intercept`, and sort the $p$-values to easily identify the most significant predictors.

In [ ]:
sorted_pvalues = mA.pvalues.drop('Intercept').sort_values()
display(sorted_pvalues)

,0
agg_inc,5.249611e-62
sqft,1.063204e-09
col_grad,2.309665e-03
com60,2.686647e-02


## Q6 – Correlation Matrix & Predictor Analysis
We calculate the correlations among all 10 store predictors to check for strong relationships (multicollinearity) and identify which variables are most closely related to the target variable, `annual_profit`.

In [ ]:
import itertools

# Identify the 10 predictors
preds = [c for c in train.columns if c not in ('store_number', Y, 'state')]

# 1. Compute and display correlation matrix rounded to 2 decimal places
corr = train[preds].corr()
print("Correlation Matrix for Predictors:")
display(corr.round(2))

# 2. Sort and display the top 5 pairs of predictors by absolute correlation value
pairs = sorted(
    ((a, b, corr.loc[a, b]) for a, b in itertools.combinations(preds, 2)),
    key=lambda t: -abs(t[2])
)
top_pairs = pd.DataFrame(pairs[:5], columns=['var1', 'var2', 'corr'])
print("\nTop 5 Most Correlated Pairs:")
display(top_pairs)

# 3. Correlation of each predictor with the outcome variable (annual_profit)
print("\nCorrelation of Predictors with Annual Profit (Y):")
display(train[preds + [Y]].corr()[Y].sort_values())

Correlation Matrix for Predictors:


,agg_inc,sqft,col_grad,com60,lci,nearcomp,nearmil,freestand,gini,housemed
agg_inc,1.00,0.51,0.67,-0.24,-0.31,-0.15,0.16,0.17,0.07,0.49
sqft,0.51,1.00,0.35,-0.06,-0.30,-0.07,0.12,0.15,0.11,0.18
col_grad,0.67,0.35,1.00,-0.22,-0.31,-0.18,0.07,0.17,0.00,0.55
com60,-0.24,-0.06,-0.22,1.00,-0.00,0.02,0.04,-0.03,-0.04,-0.11
lci,-0.31,-0.30,-0.31,-0.00,1.00,0.07,-0.11,-0.22,-0.10,-0.15
nearcomp,-0.15,-0.07,-0.18,0.02,0.07,1.00,-0.11,0.10,0.11,-0.17
nearmil,0.16,0.12,0.07,0.04,-0.11,-0.11,1.00,0.21,-0.01,0.03
freestand,0.17,0.15,0.17,-0.03,-0.22,0.10,0.21,1.00,0.02,-0.01
gini,0.07,0.11,0.00,-0.04,-0.10,0.11,-0.01,0.02,1.00,-0.08
housemed,0.49,0.18,0.55,-0.11,-0.15,-0.17,0.03,-0.01,-0.08,1.00



Top 5 Most Correlated Pairs:


,var1,var2,corr
0,agg_inc,col_grad,0.670194
1,col_grad,housemed,0.546932
2,agg_inc,sqft,0.511834
3,agg_inc,housemed,0.486390
4,sqft,col_grad,0.353035



Correlation of Predictors with Annual Profit (Y):


,annual_profit
lci,-0.407641
com60,-0.149642
nearcomp,0.038363
gini,0.095546
nearmil,0.260600
housemed,0.374731
freestand,0.485003
sqft,0.579068
col_grad,0.635193
agg_inc,0.868277


## Q7 – Fitting Model B (All 10 Variables)
We fit a comprehensive OLS regression model containing all 10 predictors (`Model B`) to evaluate the performance improvement and inspect the statistical significance ($p$-values) of the 6 newly added market and site characteristics.

In [ ]:
# Fit the full model (Model B) with all 10 predictors
mB = smf.ols(Y + ' ~ ' + ' + '.join(preds), data=train).fit()
print(mB.summary())

# Extract and display the p-values of the 6 new variables
new_vars = ['lci', 'nearcomp', 'nearmil', 'freestand', 'gini', 'housemed']
print("\np-values of the 6 new variables:")
display(mB.pvalues[new_vars].sort_values())

                            OLS Regression Results                            
Dep. Variable:          annual_profit   R-squared:                       0.918
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     406.1
Date:                Mon, 02 Nov 2026   Prob (F-statistic):          2.74e-190
Time:                        03:11:44   Log-Likelihood:                -4928.3
No. Observations:                 374   AIC:                             9879.
Df Residuals:                     363   BIC:                             9922.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   1.279e+05    6.2e+04      2.062      0.0

,0
freestand,7.175235e-51
nearcomp,1.182668e-18
nearmil,3.188841e-06
lci,4.380442e-04
housemed,3.430107e-01
gini,7.237427e-01


## Q8 – Screening Single Additions for Model D
We test adding each of our four highly significant new variables (`lci`, `nearcomp`, `nearmil`, and `freestand`) individually to the original 4 variables (`A_vars`) to find which one yields the highest out-of-sample Test $R^2$.

In [ ]:
for v in ['lci', 'nearcomp', 'nearmil', 'freestand']:
    _, r2_tr, r2_te, tot = fit_eval(A_vars + [v])
    print(f'{v:10s} train R2={r2_tr:.4f}  test R2={r2_te:.4f}  total=${tot:.2f}M')

lci        train R2=0.7968  test R2=0.7670  total=$40.10M
nearcomp   train R2=0.8179  test R2=0.7703  total=$39.97M
nearmil    train R2=0.7997  test R2=0.7161  total=$37.78M
freestand  train R2=0.8916  test R2=0.8188  total=$37.33M


### Defining Model C and Model D
Based on our rules:
1. **Model C:** Start with all 10 predictors (`preds`). Drop non-significant variables from Model B (`gini` and `housemed` had $p \ge 0.05$). Also check if there are any remaining pairs with $|\text{corr}| > 0.70$. Since our highest pairwise correlation is `agg_inc` and `col_grad` at $0.67$ (which is $<0.70$), no further columns need to be dropped due to the multicollinearity rule.
2. **Model D:** We choose the variable that improves test $R^2$ over Model A while keeping the 48-site total at or above $40M.

In [ ]:
# Model C: Drop 'gini' and 'housemed'
C_vars = [v for v in preds if v not in ['gini', 'housemed']]

# Model D: Adding the single best performer that keeps the 48-site total >= $40M (lci)
D_vars = A_vars + ['lci']

rows = []
for name, v in [('A', A_vars), ('B', preds), ('C', C_vars), ('D', D_vars)]:
    _, r2_tr, r2_te, tot = fit_eval(v)
    rows.append([name, len(v), round(r2_tr, 4), round(r2_te, 4), round(tot, 2)])

comparison_df = pd.DataFrame(rows, columns=['Model', '# vars', 'Train R2', 'Test R2', 'Total 48 sites ($M)'])
display(comparison_df)

,Model,# vars,Train R2,Test R2,Total 48 sites ($M)
0,A,4,0.7861,0.7201,40.02
1,B,10,0.9179,0.8272,36.06
2,C,8,0.9177,0.8420,36.29
3,D,5,0.7968,0.7670,40.10


# Answers

---

### 1. Dataset Preparation and Rationale

#### **Q1. Why did Kathleen’s team split the data into a training set (374 stores) and test set (85 stores)?**
* **(a) What percentage of the total data is in the training set?**
  * **Answer:** **81.5%** of the historical store data is in the training set ($374 / (374 + 85)$).
* **(b) Explain what the training set will be used for, what the test set will be used for, and why it is important not to use the test set during model building.**
  * **Answer:**
    * **Training Set:** Used to estimate (fit) the regression parameters (coefficients).
    * **Test Set:** Used to evaluate how well the trained model predicts profitability on unseen data (out-of-sample performance).
    * **Why avoid test set contamination:** If the test set is used during model building, the model's parameters will adjust to fit those specific test data points. This creates "data leakage," inflating the performance metrics and rendering it impossible to get an unbiased measure of how the model will perform on genuinely new locations.

---

### 2. Kathleen’s Original Model

#### **Q2. Write the complete linear regression equation for predicting annual store profitability from these four predictors based on your training data.**
* **Answer:** Using the coefficients calculated from our OLS output:
  $$\text{annual\_profit} = 83,600 + 0.0028 \times \text{agg\_inc} + 383.36 \times \text{sqft} + 346,800 \times \text{col\_grad} + 218,300 \times \text{com60}$$

#### **Q3. Using the estimated regression model, what annual profitability is predicted for a Milagro store located in an area with the given characteristics?**
* **Input Parameters:**
  * `agg_inc` = $100,000,000
  * `sqft` = 800
  * `col_grad` = 0.30 (30%)
  * `com60` = 0.10 (10%)
* **Answer:** **$796,894.99**

#### **Q4. Evaluate the quality of the original model:**
* **(a) What is the $R^2$ value on the training data?**
  * **Answer:** **78.61%** ($0.7861$)
* **(b) What is the $R^2$ value on the test data?**
  * **Answer:** **72.01%** ($0.7201$)

#### **Q5. Test the statistical significance of the predictors in Model A:**
* **(a) Which independent variables are statistically significant at the 5% level ($\alpha = 0.05$)?**
  * **Answer:** **All four variables** (`agg_inc`, `sqft`, `col_grad`, `com60`) are statistically significant because each has a $p$-value $< 0.05$.
* **(b) Which variable has the smallest p-value (most statistically significant)?**
  * **Answer:** **`agg_inc`** ($p = 5.25 \times 10^{-62}$)
* **(c) Which variable has the largest p-value (least statistically significant, but still below 0.05)?**
  * **Answer:** **`com60`** ($p = 0.0269$)

---

### 3. Exploratory Correlation Analysis

#### **Q6. Compute the correlation matrix for all numerical predictor variables (exclude store.number, annual.profit, and state).**
* **(a) Identify the three pairs of variables with the strongest correlations (highest absolute values). Report the correlation coefficient for each pair.**
  1. **`agg_inc` and `col_grad`**: $r = 0.67$
  2. **`col_grad` and `housemed`**: $r = 0.55$
  3. **`agg_inc` and `sqft`**: $r = 0.51$

#### **Q7. Build a regression model using ALL 10 variables (the 4 original plus 6 new variables). Test significance at the 5% level ($\alpha = 0.05$):**
* **(a) Which of the 6 new variables are statistically significant?**
  * **Answer:** **`freestand`** ($p \approx 0.00$), **`nearcomp`** ($p \approx 0.00$), **`nearmil`** ($p \approx 0.00$), and **`lci`** ($p = 0.0004$) are highly statistically significant.
* **(b) Which of the new variables are NOT significant? What does this suggest about their usefulness in predicting store profitability?**
  * **Answer:** **`gini`** ($p = 0.724$) and **`housemed`** ($p = 0.343$) are not statistically significant. This suggests that once you control for general demographic variables (like household size and regional income), inequality levels (`gini`) and median home values (`housemed`) provide no additional useful predictive signal and can be omitted to avoid overfitting.

---

### 4. Model Comparison

#### **Q8. Model Performance Table Summary**

| Model | # variables | Train $R^2$ | Test $R^2$ (OSR2) | Total 48 sites Predicted ($M) |
| :---: | :---: | :---: | :---: | :---: |
| **Model A** | 4 | 78.61% | 72.01% | **$40.02M** |
| **Model B** | 10 | 91.79% | 82.72% | **$36.06M** |
| **Model C** | 8 | 91.77% | **84.20%** | **$36.29M** |
| **Model D** | 5 | 79.68% | **76.70%** | **$40.10M** |

* **(a) For Model D report which variable you added:**
  * **Answer:** We added **`lci`**. This variable improves out-of-sample prediction accuracy (Test $R^2$ = 76.70% vs. Model A's 72.01%) while maintaining the aggregate predicted profitability for the 48 construction sites at or above the $40 million target ($40.10M).

* **(b) Summary statistics comparison:** (See table above).

---

### 5. Recommendation & The Dilemma

#### **Q9. Model recommendation and the dilemma:**

* **Answer:** I recommend Model C because it has the highest test $R^2$ (0.842), so it gives the most reliable predictions for new stores. However, it predicts about $36.3M in total profit for the 48 sites, which is below the $40M already shared with Harriman Capital. Lowering the estimate is a risk, because Harriman may cut its offer or walk away. But choosing Model D just because it reaches $40M is worse: if the stores actually earn closer to $36M, Harriman overpays, Milagro loses trust and may face legal problems, and Kathleen's reputation suffers. Being accurate and honest now protects everyone more than hitting a number that may not hold up.